# 07 — Error Analysis, Sensitivity, and Failure Cases

**Objective:** Understand *when* calibration fails, *why* reprojection error is high for some images, and how sensitive the calibration is to the number of images, board position diversity, and image quality.

This notebook answers concrete engineering questions:
- What happens if I use only 5 images instead of 20?
- Which images contribute the most error?
- What does a failed detection look like — and why?


In [ ]:
import sys, logging
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.calibrator import CameraCalibrator
from calibvision.utils import load_images_from_dir

logging.basicConfig(level=logging.WARNING)  # suppress INFO during loops

PROJECT_ROOT = Path('..').resolve()
CALIB_DIR    = PROJECT_ROOT / 'data' / 'calibration_images'
BOARD_COLS, BOARD_ROWS = 9, 6

# Load all detections once
paths, _ = load_images_from_dir(CALIB_DIR)
calibrator = CameraCalibrator(board_cols=BOARD_COLS, board_rows=BOARD_ROWS, min_valid_frames=5)
all_detections = calibrator.detect_corners(paths, verbose=False)
valid = [d for d in all_detections if d.success]
print(f'Valid detections available: {len(valid)}')

In [ ]:
# ── 7.1  Sensitivity to number of calibration images ─────────────────────────
# How does RMS error change as we add more images?
# Best practice: use all valid images, but understand diminishing returns.

np.random.seed(42)
rms_by_count = {}
min_n = 5

for n in range(min_n, len(valid) + 1):
    subset = valid[:n]  # sequential — not random, to show effect of diversity
    try:
        result = calibrator.calibrate(subset)
        rms_by_count[n] = result.rms_reprojection_error
    except ValueError:
        pass

fig, ax = plt.subplots(figsize=(10, 4))
ns = list(rms_by_count.keys())
rms_vals = list(rms_by_count.values())
ax.plot(ns, rms_vals, 'o-', color='#3498db', linewidth=2, markersize=5)
ax.axhline(1.0, color='#e74c3c', linestyle='--', label='1.0 px threshold')
ax.axhline(0.5, color='#2ecc71', linestyle='--', label='0.5 px (excellent)')
ax.set_xlabel('Number of calibration images used')
ax.set_ylabel('RMS Reprojection Error (px)')
ax.set_title('Calibration quality vs number of images\n(Diminishing returns after ~12 images)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'With {min_n} images: RMS = {rms_by_count.get(min_n, "N/A")}')
print(f'With all {len(valid)} images: RMS = {rms_by_count.get(len(valid), "N/A"):.4f} px')

In [ ]:
# ── 7.2  Leave-one-out analysis — which images hurt calibration? ──────────────
# Remove each image one at a time. A large RMS drop when removed → that image
# was poorly captured and is degrading calibration quality.

if len(valid) >= 8:
    base_result = calibrator.calibrate(valid)
    base_rms = base_result.rms_reprojection_error

    loo_rms = []
    for i in range(len(valid)):
        subset = valid[:i] + valid[i+1:]
        try:
            r = calibrator.calibrate(subset)
            loo_rms.append(r.rms_reprojection_error)
        except ValueError:
            loo_rms.append(None)

    fig, ax = plt.subplots(figsize=(12, 4))
    valid_loo = [(i, v) for i, v in enumerate(loo_rms) if v is not None]
    idxs = [x[0] for x in valid_loo]
    vals = [x[1] for x in valid_loo]

    colors = ['#e74c3c' if v < base_rms - 0.01 else '#95a5a6' for v in vals]
    ax.bar(idxs, vals, color=colors)
    ax.axhline(base_rms, color='#3498db', linewidth=2,
               label=f'Baseline RMS = {base_rms:.4f} px')
    ax.set_xlabel('Image removed (index)')
    ax.set_ylabel('RMS without that image')
    ax.set_title('Leave-one-out analysis\nRed bars: removing this image IMPROVES calibration')
    ax.legend()
    plt.tight_layout()
    plt.show()

    candidates = [(i, v) for i, v in valid_loo if v is not None and v < base_rms - 0.01]
    if candidates:
        print(f'Candidates for removal (improve RMS):')
        for i, v in sorted(candidates, key=lambda x: x[1]):
            print(f'  Remove image {i} ({valid[i].image_path.name}) → RMS = {v:.4f} px')
    else:
        print('All images contribute positively — no candidates for removal')
else:
    print('Need ≥ 8 images for leave-one-out analysis')

In [ ]:
# ── 7.3  Simulated failure cases ──────────────────────────────────────────────
# Demonstrate what prevents corner detection: blur, truncation, wrong board size.

def make_synthetic_checkerboard_base(cols=9, rows=6, square_px=60, border=60):
    bh = rows * square_px + 2 * border
    bw = cols * square_px + 2 * border
    img = np.full((bh, bw), 255, dtype=np.uint8)
    for r in range(rows + 1):
        for c in range(cols + 1):
            if (r + c) % 2 == 0:
                y0 = border + r * square_px
                x0 = border + c * square_px
                img[y0:y0 + square_px, x0:x0 + square_px] = 0
    return img

board = make_synthetic_checkerboard_base()

failure_cases = [
    ('Good (reference)',      board.copy()),
    ('Heavy blur (σ=8)',      cv2.GaussianBlur(board, (31, 31), 8)),
    ('Partial (cropped 40%)', board[:, :int(board.shape[1]*0.6)]),
    ('Wrong board (8×5)',     make_synthetic_checkerboard_base(cols=8, rows=5)),
    ('Low contrast',          cv2.convertScaleAbs(board, alpha=0.15, beta=200)),
]

fig, axes = plt.subplots(1, len(failure_cases), figsize=(18, 4))
for ax, (label, img) in zip(axes, failure_cases):
    # Try detection on THIS image with the CORRECT board size
    found, _ = cv2.findChessboardCorners(img, (BOARD_COLS, BOARD_ROWS), None)
    status = '✓ Found' if found else '✗ Failed'
    color = '#2ecc71' if found else '#e74c3c'

    ax.imshow(img, cmap='gray', aspect='auto')
    ax.set_title(f'{label}\n{status}', fontsize=8,
                 color='green' if found else 'red')
    ax.axis('off')

plt.suptitle('Common calibration failure modes', fontsize=11)
plt.tight_layout()
plt.show()

## Summary

| Analysis | Conclusion |
|---|---|
| N-image sensitivity | Diminishing returns after ~12 images; more diversity helps more than count |
| Leave-one-out | Identifies outlier images degrading calibration |
| Failure simulation | Blur, partial view, wrong board size, low contrast all cause detection failure |

## Next step

Open `08_saved_profile_reuse.ipynb` to load a saved profile on new images without re-running calibration.
